# Convergence across chains, UCI boston split 0

Four chains per sampler for the small ($d = 752$) and medium ($d = 5852$) network, each started from its own MAP with its own Laplace covariance, and the sticky samplers at $w = 0.3$ and $w = 0.1$. `BUDGET` picks $10^6$ or $10^7$ gradient evaluations per PDMP chain. At $10^7$ the medium chains are 0 to 3 (chain 0 continues the published run) and the small chains 1 to 4 (chain 4 has a fresh MAP). NUTS runs 4 chains of 1000 draws from each of the same MAPs. All PDMP chains keep $S = 4000$ draws, uniform in time after a 20 % time burn-in, in time order.

In [ ]:
import os
import sys
import warnings
from pathlib import Path

import arviz as az
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
import torch

if Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.insert(0, "notebooks")
warnings.filterwarnings("ignore", module="arviz")

from utils.convergence_utils import CONV, SAMPLERS_AT, chain_paths, nuts_paths, nuts_chains, draw_stats
from sazz.gpu_friendly.scripts.uci_bnn_grid import load_raw_datasets, make_split, BASE_SEED

DATASET, SPLIT_ID = "boston", 0
BUDGET = {"small": "1e7", "medium": "1e7"}      # gradients per PDMP chain, "1e6" or "1e7"
PAPER_DIR = Path("results/plots_v2/convergence/paper")
PAPER_DIR.mkdir(parents=True, exist_ok=True)

# Same sampler colours as the other paper figures
SAMPLERS = {
    "boomerang":        ("C0", "Boomerang"),
    "sticky_boomerang": ("C1", "Sticky Boomerang"),
    "zigzag":           ("C2", "ZigZag"),
    "sticky_zigzag":    ("C3", "Sticky ZigZag"),
    "nuts":             ("C4", "NUTS"),
}
NETWORKS = {"small": "small network ($d=752$)", "medium": "medium network ($d=5852$)"}
BUDGET_TEX = {"1e6": "$10^6$", "1e7": "$10^7$"}

X_all, y_all = load_raw_datasets((DATASET,))[DATASET]
data = make_split(X_all, y_all, seed=BASE_SEED + SPLIT_ID, dtype=torch.float64, device="cpu")

# chain_ids[key][i] is the MAP chain i started from, chains[key][i] its per-draw statistics
RUNS = [(net, w, s) for net in NETWORKS for w in (0.3, 0.1) for s in SAMPLERS_AT[w]]
chain_ids, chains = {}, {}
for key in RUNS:
    found = chain_paths(*key, DATASET, SPLIT_ID, budget=BUDGET[key[0]])
    chain_ids[key] = [c for c, _ in found]
    chains[key] = [draw_stats(p, data) for _, p in found]
# NUTS from the same MAPs as the PDMP chains of each network
nuts_maps = {net: tuple(sorted({c for k in RUNS if k[0] == net for c in chain_ids[k]})) for net in NETWORKS}
nuts = {net: [draw_stats(p, data) for p in nuts_paths(net, DATASET, SPLIT_ID, maps=nuts_maps[net])] for net in NETWORKS}
assert all(len(runs) == 4 for runs in nuts.values()), {net: len(runs) for net, runs in nuts.items()}
print("PDMP chains", {net: sorted({tuple(chain_ids[k]) for k in RUNS if k[0] == net}) for net in NETWORKS})
print("NUTS MAPs  ", nuts_maps)

In [ ]:
# Helpers shared by every figure and table below
PAPER_RC = {"font.size": 14, "axes.titlesize": 15, "axes.labelsize": 14, "xtick.labelsize": 12,
            "ytick.labelsize": 12, "legend.fontsize": 12}
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.alpha": 0.25, "figure.dpi": 120, "savefig.dpi": 300, "pdf.fonttype": 42})

# x-axis order of every figure and row order of every table, (w, sampler), NUTS has no w
PAPER_ORDER = [(0.3, "zigzag"), (0.3, "boomerang"), (None, "nuts"),
               (0.3, "sticky_zigzag"), (0.3, "sticky_boomerang"),
               (0.1, "sticky_zigzag"), (0.1, "sticky_boomerang")]
SHORT = {"zigzag": "ZigZag", "boomerang": "Boomerang", "nuts": "NUTS",
         "sticky_zigzag": "Sticky\nZigZag", "sticky_boomerang": "Sticky\nBoomerang"}


def rhat(a):
    return float(az.rhat(a, method="rank"))


def ess(a):
    return float(az.ess(a, method="bulk"))


def label(w, s):
    return f"{SAMPLERS[s][1]}, $w={w:g}$" if "sticky" in s else SAMPLERS[s][1]


def short_label(w, s):
    return SHORT[s] + (f"\n$w={w:g}$" if "sticky" in s else "")


def shades(color, n=4):
    """n shades of color, dark to light, one per chain in MAP order."""
    base = np.array(mcolors.to_rgb(color))
    return [tuple(base * (1 - a) + a * np.ones(3)) for a in np.linspace(0.0, 0.55, n)]


def load_budget(net, budget, order=PAPER_ORDER):
    """{(w, sampler): (chain ids, stats)} for one network at one budget, reusing what is loaded."""
    out = {}
    for w, s in order:
        if s == "nuts":
            continue
        if BUDGET[net] == budget and (net, w, s) in chains:
            out[w, s] = (chain_ids[net, w, s], chains[net, w, s])
        else:
            found = chain_paths(net, w, s, DATASET, SPLIT_ID, budget=budget)
            out[w, s] = ([c for c, _ in found], [draw_stats(p, data) for _, p in found])
    return out


# def chain_means_panel(ax, runs_by_key, nuts_runs, stat, order=PAPER_ORDER):
#     """Chain mean +- 1 sd per chain, the starting MAP as a hollow marker joined by a dotted line."""
#     for i, (w, s) in enumerate(order):
#         runs = nuts_runs if s == "nuts" else runs_by_key[w, s][1]
#         for c, (st, colr) in enumerate(zip(runs, shades(SAMPLERS[s][0]))):
#             x = i + (c - 1.0) * 0.17
#             m_map = st.get(f"MAP {stat}")
#             if m_map is not None:
#                 ax.plot([x, x], [m_map, st[stat].mean()], color=colr, lw=0.9, ls=":", zorder=1)
#                 ax.plot(x, m_map, marker="o", ms=7, mfc="white", mec=colr, mew=1.6, ls="none", zorder=2)
#             ax.errorbar(x, st[stat].mean(), yerr=st[stat].std(), fmt="o", ms=8, zorder=3,
#                         color=colr, ecolor=colr, elinewidth=2, capsize=0, mec="white", mew=0.7)
#     if (None, "nuts") in order:
#         k = order.index((None, "nuts"))
#         ax.axvspan(k - 0.5, k + 0.5, color="0.92", zorder=0)   # NUTS column
#         ax.axvline(k + 0.5, color="0.6", lw=0.8)               # dense | sticky
#     ax.set_xticks(range(len(order)), [short_label(w, s) for w, s in order])

CHAIN_SPREAD = 0.8      # width each sampler's group of chains takes on the x axis (1 = touching the next group)


def chain_means_panel(ax, runs_by_key, nuts_runs, stat, order=PAPER_ORDER):
    """Chain mean +- 1 sd per chain, the starting MAP as a hollow marker joined by a dotted line."""
    for i, (w, s) in enumerate(order):
        runs = nuts_runs if s == "nuts" else runs_by_key[w, s][1]
        n = len(runs)
        step = CHAIN_SPREAD / max(n, 1)
        for c, (st, colr) in enumerate(zip(runs, shades(SAMPLERS[s][0], n))):
            x = i + (c - (n - 1) / 2) * step                      # centred on the tick
            m_map = st.get(f"MAP {stat}")
            if m_map is not None:
                ax.plot([x, x], [m_map, st[stat].mean()], color=colr, lw=1.2, ls=":", zorder=1)
                ax.plot(x, m_map, marker="o", ms=9, mfc="white", mec=colr, mew=2.0, ls="none", zorder=2)
            ax.errorbar(x, st[stat].mean(), yerr=st[stat].std(), fmt="o", ms=11, zorder=3,
                        color=colr, ecolor=colr, elinewidth=3.5, capsize=0, mec="white", mew=0.9)
    if (None, "nuts") in order:
        k = order.index((None, "nuts"))
        ax.axvspan(k - 0.5, k + 0.5, color="0.92", zorder=0)   # NUTS column
        ax.axvline(k + 0.5, color="0.6", lw=0.8)               # dense | sticky
    ax.set_xlim(-0.5, len(order) - 0.5)                        # no empty margin left and right
    ax.set_xticks(range(len(order)), [short_label(w, s) for w, s in order], fontsize=16, rotation=30)


# def plot_chain_means(columns, fname, order=PAPER_ORDER):
#     """columns is a list of (title, runs_by_key, nuts_runs), one column each."""
#     with plt.rc_context(PAPER_RC):
#         fig, axes = plt.subplots(2, len(columns), figsize=(6 * len(columns), 6),
#                                  sharex=True, sharey="row", squeeze=False)
#         for col, (title, runs_by_key, nuts_runs) in enumerate(columns):
#             for row, stat in enumerate(("log sigma", "test RMSE")):
#                 chain_means_panel(axes[row, col], runs_by_key, nuts_runs, stat, order)
#                 if col == 0:
#                     axes[row, col].set_ylabel("RMSE" if stat == "test RMSE" else r"$\log\sigma$", fontsize=20)
#             if title == "small network ($d=752$), $10^7$ gradients":
#                 axes[0, col].set_title(r"$d=752$, $10^7$ gradients", fontsize=20)
#             else:
#                 axes[0, col].set_title(r"$d=5852$, $10^7$ gradients", fontsize=20)
#         handles = [Line2D([], [], color="0.3", marker="o", ms=20, ls="none", mec="white",
#                           label=r"Chain mean $\pm$ 1 sd"),
#                    Line2D([], [], color="0.3", marker="o", ms=20, mfc="white", mew=1.6, ls=":",
#                           label="Starting MAP")]
#         fig.tight_layout()
#         fig.legend(handles=handles, loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=2, frameon=False, fontsize=20)
#         fig.savefig(PAPER_DIR / fname, bbox_inches="tight")
#         plt.show()
        
STATS_PLOT = ("log sigma", "test RMSE", "test mean prediction")    # one row each
YLABEL = {"log sigma": r"$\log\sigma$", "test RMSE": "RMSE", "test mean prediction": "Mean prediction"}


def plot_chain_means(columns, fname, order=PAPER_ORDER, stats=STATS_PLOT):
    """columns is a list of (title, runs_by_key, nuts_runs), one column each."""
    with plt.rc_context(PAPER_RC):
        fig, axes = plt.subplots(len(stats), len(columns), figsize=(6 * len(columns), 3 * len(stats)),
                                 sharex=True, sharey="row", squeeze=False)
        for col, (title, runs_by_key, nuts_runs) in enumerate(columns):
            for row, stat in enumerate(stats):
                chain_means_panel(axes[row, col], runs_by_key, nuts_runs, stat, order)
                if col == 0:
                    axes[row, col].set_ylabel(YLABEL[stat], fontsize=20)
            if title == "small network ($d=752$), $10^7$ gradients":
                axes[0, col].set_title(r"$d=752$, $10^7$ gradients", fontsize=20)
            else:
                axes[0, col].set_title(r"$d=5852$, $10^7$ gradients", fontsize=20)
        handles = [Line2D([], [], color="0.3", marker="o", ms=20, ls="none", mec="white",
                          label=r"Chain mean $\pm$ 1 sd"),
                   Line2D([], [], color="0.3", marker="o", ms=20, mfc="white", mew=1.6, ls=":",
                          label="Starting MAP")]
        fig.tight_layout()
        fig.legend(handles=handles, loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=2, frameon=False, fontsize=20)
        fig.savefig(PAPER_DIR / fname, bbox_inches="tight")
        plt.show()


In [ ]:
runs_1e7 = {net: load_budget(net, "1e7") for net in NETWORKS}
runs_1e6 = {net: load_budget(net, "1e6") for net in NETWORKS}

# Paper material

### Figure, chain means for the small and medium network at $10^7$ gradient evaluations

In [ ]:
PAPER_ORDER_use = [(0.3, "boomerang"), (0.3, "zigzag"), (None, "nuts"),
                (0.3, "sticky_boomerang"), (0.3, "sticky_zigzag"),]
               #(0.1, "sticky_zigzag"), (0.1, "sticky_boomerang")]
# NETWORKS_use = {"small": "$d=752$", "medium": "$d=5852$"}
plot_chain_means([(f"{NETWORKS[net]}, {BUDGET_TEX['1e7']} gradients", runs_1e7[net], nuts[net]) for net in NETWORKS],
                 f"{DATASET}_chain_means_1e7.pdf", order=PAPER_ORDER_use)

**Caption.** Chain means of $\log\sigma$ (top) and the test RMSE (bottom) on Boston for the small (left) and medium (right) network, with $10^7$ gradient evaluations per PDMP chain. Each filled dot is one chain with a bar of one posterior standard deviation, shaded from dark to light in MAP order. The hollow dot is the MAP the chain started from. NUTS (shaded column) runs from the same four MAPs.

# Supplements

### Table, $\hat R$, ESS per gradient and the spread between chains

In [ ]:
def conv_row(runs):
    """R-hat and bulk ESS of log sigma and test RMSE pooled over the chains, and the
    bulk ESS of both per 1e7 gradient evaluations."""
    arr = lambda k: np.stack([st[k] for st in runs])
    grads = sum(st["grads"] for st in runs)
    ls, rm = arr("log sigma"), arr("test RMSE")
    return rhat(ls), rhat(rm), ess(ls), ess(rm), ess(ls) / grads * 1e7, ess(rm) / grads * 1e7


def nuts_row(runs):
    """Same for NUTS, one chain per MAP (the first of its 4), as for the PDMPs.
    Gradients are a quarter of each run's, since one chain is a quarter of the run."""
    arr = lambda k: np.stack([nuts_chains(st, k)[0] for st in runs])
    grads = sum(st["grads"] for st in runs) / 4
    ls, rm = arr("log sigma"), arr("test RMSE")
    return rhat(ls), rhat(rm), ess(ls), ess(rm), ess(ls) / grads * 1e7, ess(rm) / grads * 1e7


def fmt_ess(v):
    return f"{v:.1f}" if v < 10 else f"{v:,.0f}".replace(",", "{,}")


lines = [r"\begin{table}", r"\centering", r"\resizebox{\columnwidth}{!}{",
         r"\begin{tabular}{llccccccc}", r"\toprule",
         r" & & & \multicolumn{2}{c}{$\hat R$} & \multicolumn{2}{c}{bulk ESS} "
         r"& \multicolumn{2}{c}{bulk ESS per $10^7$ grad.\ evals} \\",
         r"\cmidrule(lr){4-5} \cmidrule(lr){6-7} \cmidrule(lr){8-9}",
         r"Network & Sampler & $w$ & $\log\sigma$ & test RMSE & $\log\sigma$ & test RMSE "
         r"& $\log\sigma$ & test RMSE \\"]
for net in NETWORKS:
    lines += [r"\midrule", rf"\multicolumn{{9}}{{l}}{{\textit{{{net} network}}}} \\"]
    for w, s in PAPER_ORDER:
        r = nuts_row(nuts[net]) if s == "nuts" else conv_row(runs_1e7[net][w, s][1])
        wtex = f"{w:g}" if "sticky" in s else "--"
        lines.append(f" & {SAMPLERS[s][1]} & {wtex} & {r[0]:.2f} & {r[1]:.2f} "
                     f"& {fmt_ess(r[2])} & {fmt_ess(r[3])} & {fmt_ess(r[4])} & {fmt_ess(r[5])} \\\\")
lines += [r"\bottomrule", r"\end{tabular}", r"}",
          r"\caption{Convergence on the first Boston split, four chains per sampler started from "
          r"different MAPs, with $10^7$ gradient evaluations per PDMP chain and one NUTS chain per MAP. "
          r"Rank-normalised $\hat R$ and bulk ESS of $\log\sigma$ and the test RMSE, pooled over the four "
          r"chains, which hold $16{,}000$ draws for the PDMPs and $4000$ for NUTS. The last two columns "
          r"are rates, the bulk ESS per $10^7$ gradient evaluations, the budget of one PDMP chain.}",
          r"\label{tab:supp_rhat}", r"\end{table}"]
print("\n".join(lines))


**Caption.** Rank-normalised $\hat R$ over four chains started from different MAPs on Boston, with $10^7$ gradient evaluations per PDMP chain, the bulk ESS of $\log\sigma$ per $10^6$ gradient evaluations, and the standard deviation of the chains' mean test RMSE. NUTS pools four chains from each of the same four MAPs.

### Figure, chain means for the medium network at $10^6$ and $10^7$ gradient evaluations

In [ ]:
medium_1e6 = load_budget("medium", "1e6")
plot_chain_means([(f"{BUDGET_TEX[b]} gradients per chain", runs, nuts["medium"])
                  for b, runs in (("1e6", medium_1e6), ("1e7", runs_1e7["medium"]))],
                 f"{DATASET}_medium_chain_means_budget.pdf")

**Caption.** Chain means of $\log\sigma$ (top) and the test RMSE (bottom) for the medium network on Boston, after $10^6$ (left) and $10^7$ (right) gradient evaluations per PDMP chain. Markers as in the main text figure.

### Figure, noise level and predictive along the run

In [ ]:
PRED_NET, PRED_BUDGET = "medium", "1e7"
PRED_ORDER = [(0.3, "zigzag"), (0.3, "boomerang"), (None, "nuts"), (0.3, "sticky_zigzag"), (0.3, "sticky_boomerang")]
y_test = (data["y_test"] * data["y_std"]).numpy()


def running_metrics(st):
    """Per-draw sigma and the test RMSE of the predictive from the first k draws, for every k."""
    f = st["f_test"]                                            # [S, N], y scale
    sig = np.exp(st["log sigma"]) * data["y_std"]
    k = np.arange(1, len(f) + 1)[:, None]
    rmse = np.sqrt(((np.cumsum(f, 0) / k - y_test) ** 2).mean(1))
    return sig, rmse


def nuts_first_chains(net):
    """The first of the 4 chains from every MAP, in the format running_metrics expects."""
    return [{"f_test": nuts_chains(st, "f_test")[0], "log sigma": nuts_chains(st, "log sigma")[0]}
            for st in nuts[net]]


def nuts_reference(net):
    """Pooled over all NUTS chains, mean sigma and the test RMSE of the predictive mean."""
    f = np.concatenate([st["f_test"] for st in nuts[net]])
    sig = np.exp(np.concatenate([st["log sigma"] for st in nuts[net]])) * data["y_std"]
    return float(sig.mean()), float(np.sqrt(((f.mean(0) - y_test) ** 2).mean()))


runs_by_key = runs_1e7[PRED_NET] if PRED_BUDGET == "1e7" else load_budget(PRED_NET, PRED_BUDGET)
nuts_ref = nuts_reference(PRED_NET)
with plt.rc_context(PAPER_RC):
    fig, axes = plt.subplots(2, len(PRED_ORDER), figsize=(4 * len(PRED_ORDER), 7), sharex=True, sharey="row")
    for col, (w, s) in enumerate(PRED_ORDER):
        runs = nuts_first_chains(PRED_NET) if s == "nuts" else runs_by_key[w, s][1]
        for st, colr in zip(runs, shades(SAMPLERS[s][0])):
            for row, m in enumerate(running_metrics(st)):
                axes[row, col].plot(np.linspace(0, 1, len(m)), m, color=colr, lw=1.2 if row == 0 else 2)
        if "sticky" not in s:      # NUTS targets the dense posterior only
            for row in range(2):
                axes[row, col].axhline(nuts_ref[row], color="k", ls="--", lw=1.5)
        axes[0, col].set_title("NUTS, first chain per MAP" if s == "nuts" else label(w, s))
        axes[1, col].set_xlabel("fraction of the run")
    axes[0, 0].set_ylabel(r"$\sigma$")
    axes[1, 0].set_ylabel("test RMSE")
    fig.tight_layout()
    fig.savefig(PAPER_DIR / f"{DATASET}_{PRED_NET}_running_predictive.pdf", bbox_inches="tight")
    plt.show()

**Caption.** Noise level $\sigma$ of every draw (top) and the test RMSE of the posterior predictive formed from the draws up to a given fraction of the run (bottom), for the medium network on Boston with $10^7$ gradient evaluations per PDMP chain. One line per chain, started from its own MAP. For NUTS, the first of the four chains from each MAP. The dashed line is NUTS pooled over all its chains, shown for the samplers that target the same posterior.

### Table, noise level at the MAP and the Boomerang reference

In [ ]:
b_runs, z_runs = runs_1e7["medium"][0.3, "boomerang"], runs_1e7["medium"][0.3, "zigzag"]
lines = [r"\begin{tabular}{cccccc}", r"\toprule",
         r"MAP & $\log\sigma$ at the MAP & mean prec. & 99\% prec. & $\log\sigma$ Boomerang & $\log\sigma$ ZigZag \\",
         r"\midrule"]
for c in b_runs[0]:
    m = torch.load(CONV["1e7"] / "medium" / f"{DATASET}/split_{SPLIT_ID:02d}" / "maps" / f"map_{c}.pt",
                   map_location="cpu", weights_only=False)
    prec = m["Sigma_inv"].double()[:-1]
    ls_b = b_runs[1][b_runs[0].index(c)]["log sigma"].mean()
    ls_z = z_runs[1][z_runs[0].index(c)]["log sigma"].mean()
    lines.append(f"{c} & {float(m['x_ref'][-1]):.2f} & {float(prec.mean()):.0f} & "
                 f"{float(torch.quantile(prec, 0.99)):.0f} & {ls_b:.2f} & {ls_z:.2f} \\\\")
lines += [r"\bottomrule", r"\end{tabular}"]
print("\n".join(lines))

**Caption.** For each MAP of the medium network on Boston, $\log\sigma$ at the MAP, the mean and 99\% quantile of the weight precision of the Boomerang reference, and the mean $\log\sigma$ of the Boomerang and ZigZag chains started there with $10^7$ gradient evaluations.

### Figure, traces

In [ ]:
with plt.rc_context(PAPER_RC):
    fig, axes = plt.subplots(2, len(PAPER_ORDER_use), figsize=(4 * len(PAPER_ORDER_use), 6.5), sharey="row")
    for row, net in enumerate(NETWORKS):
        for col, (w, s) in enumerate(PAPER_ORDER_use):
            ax = axes[row, col]
            series = ([nuts_chains(st, "log sigma")[0] for st in nuts[net]] if s == "nuts"
                      else [st["log sigma"] for st in runs_1e6[net][w, s][1]])
            for y, colr in zip(series, shades(SAMPLERS[s][0])):
                ax.plot(np.linspace(0, 1, len(y)), y, color=colr, lw=0.6, alpha=0.85)
            if row == 0:
                ax.set_title(short_label(w, s).replace("\n", " "), fontsize=20)
            if col == 0:
                if net == 'small':
                    ax.set_ylabel("$d=752$\nlog $\sigma$", fontsize=23)
                else:
                    ax.set_ylabel("$d=5852$\nlog $\sigma$", fontsize=23)
                #ax.set_ylabel(f"{net}\nTest RMSE", fontsize=20)
            if row == 1 and col == 2:
                ax.set_xlabel("Fraction of the run", fontsize=20)
    fig.tight_layout()
    fig.savefig(PAPER_DIR / f"{DATASET}_log_sigma_traces.pdf", bbox_inches="tight")
    plt.show()


with plt.rc_context(PAPER_RC):
    fig, axes = plt.subplots(2, len(PAPER_ORDER_use), figsize=(3.6 * len(PAPER_ORDER_use), 6.5), sharey="row")
    for row, net in enumerate(NETWORKS):
        for col, (w, s) in enumerate(PAPER_ORDER_use):
            ax = axes[row, col]
            series = ([nuts_chains(st, "test RMSE")[0] for st in nuts[net]] if s == "nuts"
                      else [st["test RMSE"] for st in runs_1e6[net][w, s][1]])
            for y, colr in zip(series, shades(SAMPLERS[s][0])):
                ax.plot(np.linspace(0, 1, len(y)), y, color=colr, lw=0.6, alpha=0.85)
            if row == 0:
                ax.set_title(short_label(w, s).replace("\n", " "), fontsize=20)
            if col == 0:
                if net == 'small':
                    ax.set_ylabel("$d=752$\nTest RMSE", fontsize=20)
                else:
                    ax.set_ylabel("$d=5852$\nTest RMSE", fontsize=20)
                #ax.set_ylabel(f"{net}\nTest RMSE", fontsize=20)
            if row == 1 and col == 2:
                ax.set_xlabel("Fraction of the run", fontsize=20)
    fig.tight_layout()
    fig.savefig(PAPER_DIR / f"{DATASET}_test_RMSE_traces.pdf", bbox_inches="tight")
    plt.show()
    


**Caption.** Test RMSE of every draw in time order on Boston for the small (top) and medium (bottom) network with $10^6$ gradient evaluations per PDMP chain, four chains per sampler started from different MAPs. For NUTS, the first of the four chains from each MAP.

In [ ]:
def budget_traces(key, ylabel, fname, width=4.0):
    """Each MAP's 1e7 chain (faint) under its 1e6 chain, both against the fraction of
    the run after burn-in."""
    with plt.rc_context(PAPER_RC):
        fig, axes = plt.subplots(2, len(PAPER_ORDER_use), figsize=(width * len(PAPER_ORDER_use), 6.5),
                                 sharey="row", sharex=True)
        for row, net in enumerate(NETWORKS):
            for col, (w, s) in enumerate(PAPER_ORDER_use):
                ax = axes[row, col]
                if s == "nuts":
                    layers = [([nuts_chains(st, key)[0] for st in nuts[net]], 0.85)]
                else:
                    layers = [([st[key] for st in runs_1e7[net][w, s][1]], 0.25),   # 1e7, behind
                              ([st[key] for st in runs_1e6[net][w, s][1]], 0.85)]   # 1e6, on top
                # for series, alpha in layers:
                #     for y, colr in zip(series, shades(SAMPLERS[s][0])):
                #         ax.plot(np.linspace(0, 1, len(y)), y, color=colr, lw=0.6, alpha=alpha)
                for i, (series, alpha) in enumerate(layers):
                    grey = s != "nuts" and i == 0                              # the 1e7 layer
                    for y, colr in zip(series, shades(SAMPLERS[s][0])):
                        ax.plot(np.linspace(0, 1, len(y)), y, color="0.78" if grey else colr,
                                lw=0.6, alpha=0.9 if grey else alpha)

                if row == 0:
                    ax.set_title(short_label(w, s).replace("\n", " "), fontsize=20)
                if col == 0:
                    ax.set_ylabel(f"$d={752 if net == 'small' else 5852}$\n{ylabel}", fontsize=20)
                if row == 1 and col == len(PAPER_ORDER_use) // 2:
                    ax.set_xlabel("Fraction of the run after burn-in", fontsize=20)
        fig.tight_layout()
        fig.savefig(PAPER_DIR / f"{DATASET}_{fname}_traces.pdf", bbox_inches="tight")
        plt.show()


budget_traces("log sigma", r"log $\sigma$", "log_sigma")
budget_traces("test RMSE", "Test RMSE", "test_RMSE", width=3.6)
budget_traces("test NLL", "Test NLL", "test_NLL", width=3.6)
